# Clase completa de NLP: EDA, limpieza y análisis de sentimiento con Logistic Regression

**Dataset:** `imdb_movie_reviews_large.csv`  
**Problema:** clasificación binaria de sentimiento en reseñas de películas.  
**Modelo único:** Logistic Regression.  
**Representaciones:** One-Hot / Binary Encoding, Bag of Words y TF-IDF.

---

## Objetivos de aprendizaje

Al finalizar este notebook, el estudiante estará en capacidad de:

1. Explorar visualmente un dataset textual.
2. Identificar problemas comunes en datos de texto: ruido, stop words, palabras raras, HTML, signos, longitud variable.
3. Construir una función de normalización y limpieza de texto.
4. Transformar texto en vectores usando representaciones clásicas.
5. Entrenar modelos de clasificación de sentimiento usando Logistic Regression.
6. Evaluar modelos con métricas de clasificación.
7. Comparar el impacto de diferentes representaciones y de la limpieza textual.
8. Interpretar errores y palabras importantes del modelo.

---

## Estructura del notebook

0. Configuración inicial  
1. NLP EDA  
2. Text Normalizing and Cleaning  
3. Sentiment Analysis con One-Hot / Binary Encoding  
4. Sentiment Analysis con Bag of Words  
5. Sentiment Analysis con TF-IDF  
6. Comparación global de resultados  
7. Interpretación del modelo  
8. Análisis de errores  
9. Conclusiones

> **Versión para estudiantes (sin soluciones):** las celdas marcadas `TODO` son actividades para completar en clase. Las celdas de código no incluyen resultados precalculados. Ejecute las celdas progresivamente; los bloques con `NotImplementedError` requieren implementación antes de continuar.


# 0. Configuración inicial

En esta sección preparamos las importaciones y completaremos en clase la carga del dataset y las funciones auxiliares.

En un proyecto real de NLP, esta etapa es importante porque permite que todo el pipeline sea reproducible.  
Un pipeline reproducible debe dejar claro:

- Qué datos se usan.
- Qué columnas contienen el texto y la etiqueta.
- Cómo se divide el dataset en entrenamiento y prueba.
- Qué semilla aleatoria se usa.
- Qué métricas se van a reportar.

In [ ]:
# ============================================================
# 0.1 Imports generales
# ============================================================

import os
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 160)

In [ ]:
# TODO: Defina DATA_PATH con la ruta del CSV que se usará en clase.
# Pista: confirme el nombre real del archivo antes de leerlo.
# DATA_PATH = ...

In [ ]:
# TODO: Cargue el archivo CSV en un DataFrame llamado df usando pandas.
# df = ...

In [ ]:
# TODO: Muestre las primeras 10 filas del dataset.
# ...

In [ ]:
# ============================================================
# 0.3 Inspección inicial
# ============================================================

# TODO:
# - Imprima las columnas.
# - Revise los tipos de datos.
# - Revise valores nulos.
# - Revise la distribución de la variable objetivo.

# ...

## 0.4 Definición de columnas

Este dataset tiene:

- `review`: texto de la reseña.
- `sentiment`: etiqueta de sentimiento.
    - `1`: sentimiento positivo.
    - `0`: sentimiento negativo.

Vamos a crear variables estándar para que el resto del notebook sea más fácil de reutilizar.

In [ ]:
# TODO: Identifique y defina TEXT_COL y LABEL_COL según el archivo.
# TODO: Cree la columna sentiment_label con etiquetas consistentes.
# TEXT_COL = ...
# LABEL_COL = ...
# ...

## Ejercicio 0

Responda brevemente:

1. ¿Cuántas reseñas tiene el dataset?
2. ¿Cuántas clases tiene el problema?
3. ¿El dataset está balanceado?
4. ¿Qué podría pasar si el dataset estuviera muy desbalanceado?

In [ ]:
# TODO: Calcule cuántas observaciones hay por clase.
# ...

# 1. NLP EDA

La exploración de datos en NLP no solo busca entender cuántas filas y columnas tenemos.  
En texto, también debemos entender:

- Longitud de los documentos.
- Palabras más frecuentes.
- Palabras poco frecuentes.
- Stop words.
- Diferencias entre clases.
- Ruido textual.
- Patrones visuales.

Esta etapa ayuda a decidir qué limpieza realizar y qué representación textual puede funcionar mejor.

In [ ]:
# TODO: Calcule num_chars y num_words por reseña.
# Pista: aplique len y separe el texto en palabras.
# ...

In [ ]:
# TODO: Muestre estadísticas descriptivas globales y por clase para las longitudes.
# ...

In [ ]:
# TODO: Grafique histogramas de longitud de reseñas, global y por sentimiento.
# ...

## 1.1 Palabras frecuentes

Una de las formas más simples de entender un corpus es contar palabras.

Sin embargo, hay que tener cuidado: las palabras más frecuentes suelen ser artículos, preposiciones y conectores.  
Por eso analizaremos:

1. Palabras frecuentes sin remover stop words.
2. Palabras frecuentes removiendo stop words.
3. Palabras frecuentes por clase.

In [ ]:
# TODO: Implemente estas tres funciones (no están resueltas):
# - simple_tokenize(text): tokenización simple con re.findall
# - get_top_words(texts, n=20, remove_stopwords=False): contar con Counter
# - plot_top_words(top_words, title): gráfico de barras
# Pista: ENGLISH_STOP_WORDS permite filtrar stop words.

def simple_tokenize(text):
    # TODO: complete la tokenización
    raise NotImplementedError("Completar en clase")

def get_top_words(texts, n=20, remove_stopwords=False):
    # TODO: complete el conteo de palabras
    raise NotImplementedError("Completar en clase")

def plot_top_words(top_words, title):
    # TODO: complete la visualización
    raise NotImplementedError("Completar en clase")

In [ ]:
# TODO: Compare palabras frecuentes con/sin stop words y en reviews positivas/negativas.
# Use las funciones que implementó en la celda anterior.
# ...

## 1.2 Palabras raras

Las palabras raras son términos que aparecen muy pocas veces.  
Pueden ser útiles si representan conceptos específicos, pero también pueden ser ruido:

- Errores de escritura.
- Nombres propios.
- Tokens generados por mala limpieza.
- Símbolos mezclados con texto.
- Palabras demasiado específicas.

En modelos clásicos, un vocabulario muy grande puede hacer que las matrices sean muy dispersas.

In [ ]:
# TODO: Construya el vocabulario de todo el corpus.
# Calcule palabras únicas, hapax legomena (frecuencia 1) y ejemplos.
# Pista: Counter.
# ...

In [ ]:
# TODO: Reporte el tamaño del vocabulario, cantidad y porcentaje de palabras raras.
# ...

In [ ]:
# TODO: Implemente stopword_ratio(text) y visualice su distribución.
# Pista: razón entre cantidad de stop words y total de tokens.
# ...

## 1.3 Nubes de palabras

Las nubes de palabras son visualizaciones útiles para clase porque muestran de forma intuitiva qué términos dominan en el corpus.

No deben tomarse como una prueba estadística fuerte, pero ayudan a generar hipótesis.

In [ ]:
# TODO: Genere nubes de palabras del corpus completo, reviews positivas y negativas.
# Pista: from wordcloud import WordCloud
# ...

In [ ]:
# TODO: Implemente generate_ngram_wordcloud(texts, ngram_size=2, title=None, stop_words=True).
# Pista: CountVectorizer(ngram_range=(n,n)) y WordCloud.generate_from_frequencies.
# Compare n-gramas positivos y negativos.
# ...

In [ ]:
# TODO: Explore ejemplos de reseñas de una clase.
# Revise antes el formato real de LABEL_COL y sus etiquetas.
# ...

## 1.4 Polaridad léxica exploratoria

Antes de entrenar modelos, podemos hacer una aproximación simple basada en diccionarios.

La idea es contar palabras positivas y negativas en cada review.  
Esto no reemplaza un modelo de machine learning, pero ayuda a entender el problema.

Limitaciones:

- No entiende contexto.
- No entiende sarcasmo.
- No maneja bien negaciones.
- Depende mucho de la lista de palabras.

In [ ]:
# TODO: Construya un clasificador exploratorio basado en polaridad léxica.
# 1. Obtenga palabras frecuentes por clase y compare vocabularios.
# 2. Defina una función calc_polarity(text).
# 3. Obtenga predicciones, calcule precisión, recall, F1 y accuracy.
# 4. Visualice la distribución de scores por clase.
# Evite usar información del conjunto de prueba para construir vocabularios si evalúa generalización.
# ...

## Ejercicio 1

1. Identifique las 10 palabras más frecuentes en reviews positivas y negativas.
2. ¿Hay palabras que aparecen en ambas clases?
3. ¿Qué problema puede causar una palabra muy frecuente como `movie` o `film`?
4. Encuentre 5 palabras raras y explique si parecen útiles o ruido.

# 2. Text Normalizing and Cleaning

La normalización textual busca transformar el texto a una forma más consistente.

Ejemplos de operaciones comunes:

- Pasar a minúsculas.
- Eliminar HTML.
- Eliminar URLs.
- Eliminar puntuación.
- Eliminar caracteres no alfabéticos.
- Tokenizar.
- Remover stop words.
- Manejar negaciones.
- Reducir ruido.

Importante: limpiar más no siempre significa mejorar.  
En análisis de sentimiento, algunas palabras aparentemente simples como `not`, `no` o `never` pueden cambiar completamente el significado.

Por ejemplo:

> `not good` no debería convertirse simplemente en `good`.

Por eso en este notebook vamos a conservar algunas negaciones importantes.

In [ ]:
# TODO: Construya CUSTOM_STOP_WORDS conservando negaciones relevantes para sentimiento.
# TODO: Implemente clean_text(text): minúsculas, HTML, URLs, caracteres, stop words.
# TODO: Cree df["review_clean"] y compare con el original.
# Pista: re.sub y ENGLISH_STOP_WORDS.

def clean_text(text):
    # TODO: implemente la limpieza
    raise NotImplementedError("Completar en clase")

In [ ]:
# TODO: Genere una nube de n-gramas de reseñas positivas limpias.
# ...

In [ ]:
# TODO: Genere una nube de n-gramas de reseñas negativas limpias.
# ...

In [ ]:
# TODO: Compare longitudes antes y después de limpiar.
# Calcule num_words_clean y visualice histogramas.
# ...

In [ ]:
# TODO: Grafique las palabras más frecuentes después de limpiar.
# ...

In [ ]:
# TODO: Repita el análisis exploratorio de polaridad con texto limpio.
# Compare métricas y distribución de los scores.
# ...

## 2.1 División entrenamiento / prueba

Usaremos la misma división de datos para todos los experimentos.  
Esto hace que la comparación entre métodos sea justa.

- Train: datos usados para entrenar.
- Test: datos usados para evaluar.
- Stratify: mantiene proporciones similares de clases en train y test.

In [ ]:
# TODO:
# Cree X_raw, X_clean, y.
# Divida en train y test usando train_test_split.
# Use test_size=0.2, random_state=42 y stratify=y.

# X_raw = ...
# X_clean = ...
# y = ...

# X_raw_train, X_raw_test, X_clean_train, X_clean_test, y_train, y_test = ...

## Ejercicio 2

1. Tome 5 reviews y compare manualmente el texto original contra el texto limpio.
2. ¿Qué información útil se perdió durante la limpieza?
3. ¿Qué ruido se eliminó?
4. ¿Por qué es importante conservar palabras como `not`, `no` y `never`?

# Funciones auxiliares para entrenamiento y evaluación

Antes de entrenar los modelos, vamos a definir una función reutilizable.  
Esto evita repetir código en cada representación.

Todos los experimentos usarán:

- La misma división train/test.
- El mismo modelo: Logistic Regression.
- Las mismas métricas.

In [ ]:
# TODO:
# Implemente una función train_and_evaluate que reciba:
# - vectorizer
# - X_train_text
# - X_test_text
# - y_train
# - y_test
# - experiment_name
#
# La función debe:
# 1. Ajustar el vectorizer con train.
# 2. Transformar train y test.
# 3. Entrenar LogisticRegression.
# 4. Predecir.
# 5. Calcular Accuracy, Precision, Recall, F1 y ROC-AUC.
# 6. Retornar un diccionario con resultados y objetos útiles.

# def train_and_evaluate(...):
#     ...

# 3. Sentiment Analysis con One-Hot / Binary Encoding

En texto, una forma simple de representar documentos es indicar si una palabra aparece o no aparece.

Ejemplo:

| review | amazing | boring | movie |
|---|---:|---:|---:|
| `amazing movie` | 1 | 0 | 1 |
| `boring movie` | 0 | 1 | 1 |

En este notebook usaremos `CountVectorizer(binary=True)`.  
Técnicamente, para documentos completos esto se conoce mejor como **binary bag-of-words** o **multi-hot encoding**, porque cada documento puede activar muchas palabras.

La idea es enseñar el principio de presencia/ausencia de palabras.

In [ ]:
# TODO:
# Entrene dos modelos:
# 1. Binary Encoding con texto original.
# 2. Binary Encoding con texto limpio.
#
# Use CountVectorizer(binary=True, max_features=5000)

# binary_raw_result = ...
# binary_clean_result = ...

# all_results.extend([...])

## Ejercicio 3

1. Explique con sus palabras qué significa que una matriz textual sea dispersa.
2. ¿Qué diferencia hay entre contar si una palabra aparece y contar cuántas veces aparece?
3. Compare los resultados del Binary Encoding con texto original y texto limpio.
4. ¿La limpieza mejoró el resultado? Explique una posible razón.

# 4. Sentiment Analysis con Bag of Words

Bag of Words representa cada documento contando cuántas veces aparece cada palabra.

A diferencia del binary encoding, BoW sí conserva frecuencia.

Ejemplo:

| review | amazing | boring | movie |
|---|---:|---:|---:|
| `amazing amazing movie` | 2 | 0 | 1 |
| `boring movie` | 0 | 1 | 1 |

Limitación principal:  
BoW ignora el orden de las palabras.  
Por ejemplo, no distingue bien entre:

- `good`
- `not good`

si no se maneja adecuadamente el contexto.

In [ ]:
# TODO:
# Entrene dos modelos:
# 1. BoW con texto original.
# 2. BoW con texto limpio.
#
# Use CountVectorizer(binary=False, max_features=5000)

# bow_raw_result = ...
# bow_clean_result = ...

# all_results.extend([...])

In [ ]:
# TODO:
# Cree una tabla parcial comparando Binary Encoding vs BoW.

# ...

## Ejercicio 4

1. ¿Por qué BoW puede funcionar mejor que Binary Encoding?
2. ¿En qué casos la frecuencia de una palabra puede ser útil?
3. ¿En qué casos la frecuencia puede ser engañosa?
4. Compare BoW con Binary Encoding usando F1-score.

# 5. Sentiment Analysis con TF-IDF

TF-IDF significa **Term Frequency - Inverse Document Frequency**.

La intuición es:

- Una palabra es importante si aparece varias veces en un documento.
- Pero pierde importancia si aparece en demasiados documentos.

Por eso TF-IDF ayuda a reducir el peso de palabras demasiado comunes.

Ejemplo:  
`movie` puede aparecer en muchas reseñas, pero no necesariamente indica sentimiento.  
`excellent`, `awful`, `boring` o `wonderful` pueden ser más informativas.

In [ ]:
# TODO:
# Entrene dos modelos:
# 1. TF-IDF con texto original.
# 2. TF-IDF con texto limpio.
#
# Use TfidfVectorizer(max_features=5000)

# tfidf_raw_result = ...
# tfidf_clean_result = ...

# all_results.extend([...])

## Ejercicio 5

1. Explique la diferencia entre BoW y TF-IDF.
2. ¿Por qué TF-IDF puede reducir el impacto de palabras muy comunes?
3. Compare TF-IDF contra BoW usando F1-score y ROC-AUC.
4. ¿El texto limpio fue mejor que el texto original?

# 6. Comparación global de resultados

Ahora consolidamos todos los experimentos en una sola tabla.

Esto nos permite responder:

- ¿Qué representación funcionó mejor?
- ¿La limpieza ayudó?
- ¿Cuál método tuvo mejor F1-score?
- ¿Cuál método tuvo mejor ROC-AUC?
- ¿Hay diferencias relevantes entre precision y recall?

In [ ]:
# TODO:
# Cree un DataFrame con las métricas de all_results.
# Ordene por F1 de mayor a menor.
# Grafique las métricas principales.

# results_df = ...
# results_df

In [ ]:
# TODO:
# Grafique accuracy, precision, recall, f1 y roc_auc para cada experimento.

# ...

In [ ]:
# TODO:
# Identifique el mejor experimento según F1-score.

# best_experiment = ...
# best_experiment

## Ejercicio 6

1. ¿Cuál fue el mejor método según F1-score?
2. ¿Cuál fue el mejor método según ROC-AUC?
3. ¿La limpieza mejoró todos los modelos?
4. ¿Qué modelo escogería para producción y por qué?

# 7. Interpretación del modelo

Una ventaja de Logistic Regression con representaciones clásicas es que podemos inspeccionar los coeficientes.

En clasificación binaria:

- Coeficientes positivos altos empujan hacia la clase positiva.
- Coeficientes negativos bajos empujan hacia la clase negativa.

Esto permite responder:

> ¿Qué palabras aprendió el modelo como señales de sentimiento positivo o negativo?

In [ ]:
# TODO:
# Usando el mejor modelo:
# - Obtenga los nombres de las features.
# - Obtenga los coeficientes de Logistic Regression.
# - Muestre las 20 palabras más positivas y 20 más negativas.

# ...

In [ ]:
# TODO:
# Grafique las palabras positivas y negativas más importantes.

# ...

## Ejercicio 7

1. Revise las 20 palabras más positivas.
2. Revise las 20 palabras más negativas.
3. ¿Todas tienen sentido semántico?
4. ¿Encuentra alguna palabra inesperada?
5. ¿Qué nos dice esto sobre la interpretabilidad de modelos clásicos?

# 8. Análisis de errores

Las métricas agregadas son importantes, pero no son suficientes.

Un buen análisis de NLP debe revisar ejemplos concretos donde el modelo falla:

- Falsos positivos: reviews negativas clasificadas como positivas.
- Falsos negativos: reviews positivas clasificadas como negativas.
- Casos ambiguos.
- Textos con sarcasmo.
- Textos con negaciones.
- Textos demasiado cortos o demasiado largos.

Este análisis es clave para mejorar el pipeline.

In [ ]:
# TODO:
# Construya un DataFrame con:
# - texto original
# - texto limpio
# - etiqueta real
# - predicción
# - probabilidad de clase positiva
# - tipo de error

# ...

In [ ]:
# TODO:
# Muestre ejemplos de falsos positivos y falsos negativos.

# ...

In [ ]:
# TODO:
# Encuentre predicciones de baja confianza.
# Pista: probabilidad cercana a 0.5.

# ...

## Ejercicio 8

1. Revise 5 falsos positivos.
2. Revise 5 falsos negativos.
3. ¿Qué patrones encuentra?
4. ¿Los errores parecen deberse a limpieza, representación o ambigüedad del lenguaje?
5. Proponga una mejora al pipeline.

# 9. Conclusiones y próximos pasos

En este notebook construimos un pipeline completo de análisis de sentimiento usando técnicas clásicas de NLP.

## Lo aprendido

1. La exploración textual permite entender longitud, vocabulario, ruido y diferencias entre clases.
2. La limpieza textual puede ayudar, pero también puede eliminar información útil.
3. Binary Encoding representa presencia/ausencia de palabras.
4. Bag of Words conserva frecuencia.
5. TF-IDF pondera la importancia relativa de cada palabra.
6. Logistic Regression es un modelo fuerte, simple e interpretable para clasificación de texto clásica.
7. Las métricas deben complementarse con análisis de errores.
8. La interpretación de coeficientes ayuda a entender qué aprendió el modelo.

## Limitaciones de los métodos clásicos

Estos métodos no capturan bien:

- Orden de palabras.
- Sarcasmo.
- Contexto largo.
- Significado profundo.
- Relaciones semánticas.
- Ambigüedad lingüística.

## Próximos pasos recomendados

Para continuar la clase o el proyecto, se puede explorar:

- N-gramas.
- Comparación entre stop words con y sin negaciones.
- Lemmatization con spaCy.
- Word embeddings.
- Modelos basados en transformers.
- Fine-tuning de BERT para análisis de sentimiento.

## Ejercicio final

Construya una conclusión ejecutiva de máximo 10 líneas respondiendo:

1. ¿Cuál fue el mejor modelo?
2. ¿Qué representación textual funcionó mejor?
3. ¿La limpieza ayudó?
4. ¿Qué limitaciones observó?
5. ¿Qué recomendaría como siguiente experimento?

# Anexo opcional: ideas para extender el notebook

Si se desea convertir este notebook en una práctica más avanzada, se pueden agregar:

1. `ngram_range=(1,2)` para incluir bigramas como `not good`.
2. Comparación entre remover y conservar negaciones.
3. Validación cruzada.
4. Búsqueda de hiperparámetros con `GridSearchCV`.
5. Exportación del mejor modelo con `joblib`.
6. Construcción de una función `predict_sentiment(text)`.

In [ ]:
# TODO OPCIONAL:
# Construya una función predict_sentiment(text) usando el mejor modelo.

# def predict_sentiment(text):
#     ...